<a href="https://colab.research.google.com/github/zombimann/Mathematical-video-animations-and-visualization/blob/main/_kids_high_school_derivatives_and_slopes.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Visualizing the Derivative: A Mobile-First Math Journey

### Overview for Educators & Parents
This notebook generates a high-fidelity mathematical animation designed specifically for mobile-first social media platforms (9:16 aspect ratio). The video guides students through one of the most fundamental shifts in Calculus: moving from an average rate of change (Secant) to an instantaneous rate of change (Tangent).

**Key Concepts Explored:**
*   **The Secant Line:** Visualizing the slope between two distant points P and Q.
*   **The Limit Process:** Watching point Q travel along the curve f(x)=x^2 until it practically merges with P.
*   **The Tangent Line:** Defining the derivative as the precise slope at a single point.
*   **Local Linearity (The 'Zoom' Secret):** A powerful visual proof showing that even the curviest functions look like straight lines when you look close enough. This is a core intuition for understanding why derivatives work.

---

### Project Metadata
*   **Author:** Mugambi Ndwiga
*   **Instagram:** [@MugambiNdwiga_math](https://www.instagram.com/MugambiNdwiga_math)
*   **Repository:** [GitHub: Mathematical video animations and visualization](https://github.com/zombimann/Mathematical-video-animations-and-visualization)

### Setup & Rendering
The following cells install the necessary Manim environment and LaTeX dependencies to render the animation directly in Google Colab.

In [ ]:
# 1. Update package list
!sudo apt-get update

# 2. Install Manim system dependencies + LaTeX suite
!sudo apt-get install -y libcairo2-dev libpango1.0-dev ffmpeg \
    texlive texlive-latex-extra texlive-fonts-extra \
    texlive-latex-recommended texlive-science dvisvgm

# 3. Pin NumPy to avoid Manim/Numpy 2.0 compatibility issues
!pip install "manim>=0.18.0" "numpy<2.0.0"

In [56]:
%%manim -qm -r 720,1280 DerivativeTangentZoom

"""
Module: DerivativeTangentZoom
Author: Mugambi Ndwiga (@MugambiNdwiga_math)

This script creates a mobile-optimized mathematical animation illustrating the
concept of a derivative. It visualizes the transition from a secant line
to a tangent line and demonstrates local linearity through a high-factor zoom.
"""

from manim import *
import numpy as np

# Color Palette and Configuration
BACKGROUND_COLOR   = "#0F1115"
CURVE_COLOR        = "#00F0FF"  # Cyan for the main function
SECANT_COLOR       = "#FFB347"  # Orange for the secant line
TANGENT_COLOR      = "#FF66CC"  # Pink for the derivative/tangent
TEXT_COLOR         = "#FFFFFF"

# Animation Parameters
TANGENT_X          = 1.0        # The x-coordinate where the derivative is calculated
X_START_SECANT     = 2.8        # Initial position for point Q
SECANT_STEPS       = 8          # Smoothness of the limit transition
ZOOM_FACTOR        = 25         # Magnification for local linearity demonstration

def f(x):
    """The target function: f(x) = x^2."""
    return x ** 2

def df(x):
    """The derivative of the function: f'(x) = 2x."""
    return 2 * x

class DerivativeTangentZoom(Scene):
    """
    A Manim Scene that visualizes the limit definition of a derivative.
    Optimized for vertical mobile screens (9:16 aspect ratio).
    """
    FONT_SIZE = 42
    STROKE = 6

    def construct(self):
        config.background_color = BACKGROUND_COLOR

        # --- Section 1: Title Card ---
        title = Text("THE DERIVATIVE", weight=BOLD, font_size=32, color=CURVE_COLOR).to_edge(UP, buff=0.4)
        underline = Line(LEFT, RIGHT, color=CURVE_COLOR).scale(2).next_to(title, DOWN, buff=0.1)
        header = VGroup(title, underline)
        self.add(header)

        # --- Section 2: Coordinate System ---
        axes = Axes(
            x_range=[-0.2, 3.5, 1],
            y_range=[-0.5, 10, 2],
            x_length=6,
            y_length=8,
            axis_config={"include_numbers": True, "stroke_width": 3, "font_size": 30}
        ).shift(DOWN * 0.5)

        curve = axes.plot(f, x_range=[0, 3.2], color=CURVE_COLOR, stroke_width=self.STROKE)

        # --- Section 3: The Hook ---
        hook = Text("What is a slope on a curve?", font_size=40, weight=BOLD).move_to(UP*2)
        self.play(Write(hook))
        self.wait(1)
        self.play(FadeOut(hook), Create(axes), Create(curve))

        # --- Section 4: Secant Visualization ---
        p1 = axes.c2p(TANGENT_X, f(TANGENT_X))
        dot_p = Dot(p1, color=TANGENT_COLOR, radius=0.15)
        lbl_p = Text("P", weight=BOLD).next_to(dot_p, LEFT, buff=0.2)

        x2 = X_START_SECANT
        p2 = axes.c2p(x2, f(x2))
        dot_q = Dot(p2, color=SECANT_COLOR, radius=0.15)
        lbl_q = Text("Q", weight=BOLD).next_to(dot_q, RIGHT, buff=0.2)

        secant = self._get_line(axes, TANGENT_X, x2, SECANT_COLOR)
        secant_text = Text("Secant: Slope between 2 points", font_size=34, color=SECANT_COLOR).to_edge(UP, buff=1.5)

        self.play(FadeIn(dot_p, lbl_p), FadeIn(dot_q, lbl_q))
        self.play(Create(secant), Write(secant_text))
        self.wait(1)

        # --- Section 5: Moving towards the Limit ---
        move_text = Text("Bring Q closer to P...", font_size=38, weight=BOLD).to_edge(UP, buff=1.5)
        self.play(ReplacementTransform(secant_text, move_text))

        x2_vals = np.linspace(X_START_SECANT, TANGENT_X + 0.01, SECANT_STEPS)
        for val in x2_vals:
            new_secant = self._get_line(axes, TANGENT_X, val, SECANT_COLOR)
            new_q_pos = axes.c2p(val, f(val))
            self.play(
                Transform(secant, new_secant),
                dot_q.animate.move_to(new_q_pos),
                lbl_q.animate.next_to(new_q_pos, RIGHT, buff=0.1),
                run_time=0.4
            )

        # --- Section 6: Tangent/Derivative Discovery ---
        tangent = self._get_tangent(axes, TANGENT_X, TANGENT_COLOR)
        final_text = Text("This is the Tangent!", font_size=44, color=TANGENT_COLOR, weight=BOLD).to_edge(UP, buff=1.5)
        deriv_val = Text("Slope = Derivative", font_size=40, color=TANGENT_COLOR).next_to(final_text, DOWN)

        self.play(
            FadeOut(dot_q, lbl_q),
            ReplacementTransform(secant, tangent),
            ReplacementTransform(move_text, final_text),
            Write(deriv_val)
        )
        self.wait(1)

        # --- Section 7: Local Linearity Zoom ---
        self.play(FadeOut(final_text), FadeOut(deriv_val))
        zoom_msg = Text("Crucial Secret:", font_size=42, weight=BOLD, color=YELLOW).to_edge(UP, buff=1.5)
        self.play(Write(zoom_msg))

        zoom_center = p1
        zoom_group = VGroup(axes, curve, tangent, dot_p, lbl_p)

        self.play(
            zoom_group.animate.scale(ZOOM_FACTOR, about_point=zoom_center),
            dot_p.animate.scale(1/ZOOM_FACTOR, about_point=zoom_center),
            lbl_p.animate.scale(1/ZOOM_FACTOR, about_point=zoom_center),
            run_time=4,
            rate_func=linear
        )

        final_insight = Text(
            "Smooth curves become straight\nlines if you zoom in enough!",
            font_size=38,
            weight=BOLD,
            line_spacing=1.2
        ).to_edge(DOWN, buff=1.5).set_stroke(BLACK, 8, background=True)

        self.play(Write(final_insight))
        self.wait(5)

        # --- Section 8: Branding ---
        watermark = Text("@MugambiNdwiga_math", font_size=20, weight=BOLD, color=WHITE).set_opacity(0.5).to_corner(DR)
        self.add(watermark)

    def _get_line(self, axes, x1, x2, color):
        """Helper to generate a secant line between two points on f(x)."""
        slope = (f(x2) - f(x1)) / (x2 - x1)
        return axes.plot(lambda x: slope * (x - x1) + f(x1), color=color, stroke_width=self.STROKE)

    def _get_tangent(self, axes, x0, color):
        """Helper to generate a tangent line at a specific point on f(x)."""
        slope = df(x0)
        return axes.plot(lambda x: slope * (x - x0) + f(x0), color=color, stroke_width=self.STROKE + 2)

Manim Community v0.18.1

[05/24/26 10:50:48] INFO     Animation 0 : Using cached data (hash :                           ]8;id=222051;file:///usr/local/lib/python3.12/dist-packages/manim/renderer/cairo_renderer.py\cairo_renderer.py]8;;\:]8;id=995766;file:///usr/local/lib/python3.12/dist-packages/manim/renderer/cairo_renderer.py#88\88]8;;\
                             961655326_1538480526_3069282800)                                                      

INFO:manim:Animation 0 : Using cached data (hash : 961655326_1538480526_3069282800)


[05/24/26 10:50:50] INFO     Animation 1 : Partial movie file written in                   ]8;id=406452;file:///usr/local/lib/python3.12/dist-packages/manim/scene/scene_file_writer.py\scene_file_writer.py]8;;\:]8;id=473429;file:///usr/local/lib/python3.12/dist-packages/manim/scene/scene_file_writer.py#527\527]8;;\
                             '/content/media/videos/content/1280p30/partial_movie_files/De                         
                             rivativeTangentZoom/1514079881_2268332985_663464056.mp4'                              

INFO:manim:Animation 1 : Partial movie file written in '/content/media/videos/content/1280p30/partial_movie_files/DerivativeTangentZoom/1514079881_2268332985_663464056.mp4'


...

INFO:manim:Combining to Movie file.


                    INFO                                                                   ]8;id=695390;file:///usr/local/lib/python3.12/dist-packages/manim/scene/scene_file_writer.py\scene_file_writer.py]8;;\:]8;id=407698;file:///usr/local/lib/python3.12/dist-packages/manim/scene/scene_file_writer.py#737\737]8;;\
                             File ready at                                                                         
                             '/content/media/videos/content/1280p30/DerivativeTangentZoom.                         
                             mp4'                                                                                  
                                                                                                                   

INFO:manim:
File ready at '/content/media/videos/content/1280p30/DerivativeTangentZoom.mp4'



                    INFO     Rendered DerivativeTangentZoom                                            ]8;id=317709;file:///usr/local/lib/python3.12/dist-packages/manim/scene/scene.py\scene.py]8;;\:]8;id=477525;file:///usr/local/lib/python3.12/dist-packages/manim/scene/scene.py#247\247]8;;\
                             Played 22 animations                                                                  

INFO:manim:Rendered DerivativeTangentZoom
Played 22 animations


In [ ]:
import glob
import os
from IPython.display import Video, display

# Search for the latest rendered video in the media folder
video_files = glob.glob('media/videos/**/*.mp4', recursive=True) + glob.glob('media/jupyter/*.mp4')

if video_files:
    # Get the most recently modified video file
    latest_video = max(video_files, key=os.path.getmtime)
    print(f"Displaying latest video: {latest_video}")
    display(Video(latest_video, embed=True, width=800))
else:
    print("No rendered videos found. Please ensure the Manim cell has finished executing.")